In [1]:
from langchain_ollama import ChatOllama
from typing import TypedDict, Literal
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field



In [2]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=200)

In [3]:
class TicketCategorySchema(BaseModel):
    category: Literal['Delivery Issue','Payment Issue','Product Defect','General Inquiry'] = Field(description="category of the customer support ticket")

class PrioritySchema(BaseModel):
    priority: Literal['High','Medium','Low'] = Field(description="priority level of the customer support ticket")

class TicketState(TypedDict):
    ticket : str
    summary: str
    category: Literal['Delivery Issue','Payment Issue','Product Defect','General Inquiry']
    priority: Literal['High','Medium','Low']
    

In [4]:
def ticket_summarizer(state:TicketState):
    prompt = f'generate a small summary in 1 or 2 sentences of the below customer issue. \n Customer issue - {state['ticket']}.'
    res = llm.invoke(prompt)
    return {'summary':res.content}

def ticker_classifier(state:TicketState):
    structured_response_llm = llm.with_structured_output(TicketCategorySchema)
    prompt = f'find the category of the ticket - {state['ticket']}'
    res = structured_response_llm.invoke(prompt)
    return {'category': res.category}

def priority_assignment(state:TicketState):
    structured_response_llm = llm.with_structured_output(PrioritySchema)
    prompt = f'find the priority for the ticket - {state['ticket']}'
    res = structured_response_llm.invoke(prompt)
    return {'priority': res.priority}


In [5]:
graph = StateGraph(TicketState)
graph.add_node('node_1',ticket_summarizer)
graph.add_node('node_2',ticker_classifier)
graph.add_node('node_3',priority_assignment)

graph.add_edge(START,'node_1')
graph.add_edge('node_1','node_2')
graph.add_edge('node_2','node_3')
graph.add_edge('node_3',END)
workflow = graph.compile()

In [6]:
print(workflow.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | node_1 |   
  +--------+   
      *        
      *        
      *        
  +--------+   
  | node_2 |   
  +--------+   
      *        
      *        
      *        
  +--------+   
  | node_3 |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [7]:
query = "I ordered a laptop last week  but it still has not deliverd. The tracking page has not updated for 3 days , and I am worreid the package is lost."
res = workflow.invoke({'ticket':query})

In [8]:
workflow.invoke({'ticket':query})

{'ticket': 'I ordered a laptop last week  but it still has not deliverd. The tracking page has not updated for 3 days , and I am worreid the package is lost.',
 'summary': "The customer's laptop order from last week remains unfulfilled, with no updates to the delivery status on the tracking page, causing concern that the package might be lost.",
 'category': 'Delivery Issue',
 'priority': 'High'}

In [9]:
for i in workflow.stream({'ticket':query}):
    print(i)

{'node_1': {'summary': "The customer's laptop order from a few weeks ago remains unfulfilled, with no updates on delivery status or tracking information available for three days, leading to concerns that the package may be lost."}}
{'node_2': {'category': 'Delivery Issue'}}
{'node_3': {'priority': 'High'}}
